# Stage 2: Baseline Tokenization

This notebook establishes baseline tokenization behavior on PubMedQA.

Tokenizers compared:
- `bert-base-uncased` (WordPiece, general domain)
- `dmis-lab/biobert-base-cased-v1.1` (biomedical domain)
- `gpt2` (BPE)

Computed metrics:
- average number of tokens per sample,
- fertility (tokens per word),
- UNK rate,
- average token length in characters.

In [8]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from datasets import load_dataset
from IPython.display import display
from transformers import BertTokenizer, GPT2Tokenizer

In [9]:
RANDOM_SEED = 42
SAMPLE_SIZE = 300

TOKENIZER_SPECS = {
    "bert_base": "bert-base-uncased",
    "biobert": "dmis-lab/biobert-base-cased-v1.1",
    "gpt2": "gpt2",
}

TARGET_TERMS = [
    "nephrolithiasis",
    "gastroesophageal",
    "dermographism",
    "electrocardiogram",
    "immunohistochemistry",
    "acetylcholinesterase",
    "hypercholesterolemia",
    "hepatocellular",
    "neurodegenerative",
    "myocardial",
]

In [10]:
def get_context_text(context_field):
    if isinstance(context_field, dict):
        contexts = context_field.get("contexts", [])
        if isinstance(contexts, list):
            return " ".join(str(part) for part in contexts)
    if isinstance(context_field, list):
        return " ".join(str(part) for part in context_field)
    return str(context_field)


def count_words(text):
    return len(text.split())


def get_unk_token_candidates(tokenizer):
    candidates = set()
    if tokenizer.unk_token is not None:
        candidates.add(tokenizer.unk_token)
    if tokenizer.unk_token_id is not None:
        unk_from_id = tokenizer.convert_ids_to_tokens([tokenizer.unk_token_id])[0]
        candidates.add(unk_from_id)
    return candidates


def clean_token(token):
    return re.sub(r"^[\W_]+|[\W_]+$", "", token)


def evaluate_tokenizer(tokenizer, texts):
    unk_candidates = get_unk_token_candidates(tokenizer)
    token_counts = []
    word_counts = []
    unk_count = 0
    total_tokens = 0
    char_lengths = []
    vocab_counter = Counter()

    for text in texts:
        words = count_words(text)
        encoding = tokenizer(text, add_special_tokens=True)
        ids = encoding["input_ids"]
        tokens = tokenizer.convert_ids_to_tokens(ids)
        token_counts.append(len(tokens))
        word_counts.append(words)

        for token in tokens:
            total_tokens += 1
            vocab_counter[token] += 1
            if token in unk_candidates:
                unk_count += 1
            if token not in tokenizer.all_special_tokens:
                normalized = clean_token(token)
                if normalized:
                    char_lengths.append(len(normalized))

    avg_tokens = float(np.mean(token_counts))
    fertility = float(np.sum(token_counts) / max(np.sum(word_counts), 1))
    unk_rate = float(unk_count / max(total_tokens, 1))
    avg_token_length = float(np.mean(char_lengths)) if char_lengths else 0.0

    return {
        "avg_tokens_per_sample": avg_tokens,
        "fertility": fertility,
        "unk_rate": unk_rate,
        "avg_token_length_chars": avg_token_length,
        "top_tokens": vocab_counter.most_common(30),
    }


def token_split_examples(tokenizer, terms):
    rows = []
    for term in terms:
        ids = tokenizer(term, add_special_tokens=False)["input_ids"]
        pieces = tokenizer.convert_ids_to_tokens(ids)
        rows.append({"term": term, "pieces": pieces, "n_pieces": len(pieces)})
    return pd.DataFrame(rows)


In [11]:
dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled")["train"]

df = pd.DataFrame({
    "question": dataset["question"],
    "context_text": [get_context_text(x) for x in dataset["context"]],
    "label": dataset["final_decision"],
})

df["text"] = df["question"].fillna("") + " [SEP] " + df["context_text"].fillna("")
sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
sample_texts = sample_df["text"].tolist()

print(f"Total labeled samples: {len(df)}")
print(f"Evaluation sample size: {len(sample_df)}")

Total labeled samples: 1000
Evaluation sample size: 300


In [12]:
tokenizers = {
    "bert_base": BertTokenizer.from_pretrained(TOKENIZER_SPECS["bert_base"]),
    "biobert": BertTokenizer.from_pretrained(TOKENIZER_SPECS["biobert"]),
    "gpt2": GPT2Tokenizer.from_pretrained(TOKENIZER_SPECS["gpt2"]),
}
list(tokenizers.keys())

['bert_base', 'biobert', 'gpt2']

In [13]:
metrics_rows = []
token_split_tables = []
top_tokens_json = {}

for name, tokenizer in tokenizers.items():
    metrics = evaluate_tokenizer(tokenizer, sample_texts)
    metrics_rows.append({
        "tokenizer_name": name,
        "model_id": TOKENIZER_SPECS[name],
        "avg_tokens_per_sample": metrics["avg_tokens_per_sample"],
        "fertility": metrics["fertility"],
        "unk_rate": metrics["unk_rate"],
        "avg_token_length_chars": metrics["avg_token_length_chars"],
        "stage": "stage2_baseline_tokenization",
    })
    top_tokens_json[name] = metrics["top_tokens"]

    term_table = token_split_examples(tokenizer, TARGET_TERMS)
    term_table["tokenizer_name"] = name
    token_split_tables.append(term_table)

metrics_df = pd.DataFrame(metrics_rows).sort_values(by="fertility")
examples_df = pd.concat(token_split_tables, ignore_index=True)

metrics_df

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (613 > 512). Running this sequence through the model will result in indexing errors


,tokenizer_name,model_id,avg_tokens_per_sample,fertility,unk_rate,avg_token_length_chars,stage
2,gpt2,gpt2,313.383333,1.492136,0.0,5.010264,stage2_baseline_tokenization
0,bert_base,bert-base-uncased,323.886667,1.542146,0.0,4.287131,stage2_baseline_tokenization
1,biobert,dmis-lab/biobert-base-cased-v1.1,339.490000,1.616439,0.0,4.052037,stage2_baseline_tokenization


In [15]:
print("Stage 2 baseline metrics:")
display(metrics_df)

print("\nToken split examples by tokenizer:")
for tokenizer_name in metrics_df["tokenizer_name"].tolist():
    print(f"\n{tokenizer_name}")
    section = examples_df[examples_df["tokenizer_name"] == tokenizer_name][["term", "pieces", "n_pieces"]]
    display(section)

print("\nTop 15 frequent tokens by tokenizer:")
for tokenizer_name in metrics_df["tokenizer_name"].tolist():
    print(f"\n{tokenizer_name}")
    top_df = pd.DataFrame(top_tokens_json[tokenizer_name], columns=["token", "count"])
    display(top_df.head(15))

Stage 2 baseline metrics:


,tokenizer_name,model_id,avg_tokens_per_sample,fertility,unk_rate,avg_token_length_chars,stage
2,gpt2,gpt2,313.383333,1.492136,0.0,5.010264,stage2_baseline_tokenization
0,bert_base,bert-base-uncased,323.886667,1.542146,0.0,4.287131,stage2_baseline_tokenization
1,biobert,dmis-lab/biobert-base-cased-v1.1,339.490000,1.616439,0.0,4.052037,stage2_baseline_tokenization



Token split examples by tokenizer:

gpt2


,term,pieces,n_pieces
20,nephrolithiasis,"[n, eph, rol, ith, iasis]",5
21,gastroesophageal,"[g, ast, ro, es, oph, age, al]",7
22,dermographism,"[der, m, ograph, ism]",4
23,electrocardiogram,"[elect, ro, card, i, ogram]",5
24,immunohistochemistry,"[im, mun, oh, ist, ochemistry]",5
25,acetylcholinesterase,"[acet, yl, ch, ol, ines, ter, ase]",7
26,hypercholesterolemia,"[hyper, ch, olester, ole, mia]",5
27,hepatocellular,"[he, p, ato, cell, ular]",5
28,neurodegenerative,"[ne, uro, deg, ener, ative]",5
29,myocardial,"[my, ocard, ial]",3



bert_base


...

,token,count
0,.,3921
1,Ġof,2500
2,",",2404
3,Ġthe,2370
4,Ġand,2114
5,Ġ(,1657
6,-,1601
7,Ġin,1425
8,Ġto,1081
9,Ġwith,909



bert_base


,token,count
0,.,4469
1,the,2905
2,",",2772
3,of,2540
4,and,2123
5,),1763
6,(,1748
7,-,1728
8,in,1594
9,to,1195



biobert


,token,count
0,.,4469
1,the,2905
2,",",2772
3,of,2540
4,and,2123
5,),1763
6,(,1748
7,-,1728
8,in,1717
9,to,1223


## Interpretation Checklist

When reviewing Stage 2 outputs, compare:
- fertility differences across tokenizers,
- whether biomedical terms are split into many pieces,
- UNK behavior,
- average token length and top frequent token patterns.